# A3.3.5: Database views

*IB Computer Science 2027, topic A3 Databases, A3.3 Database programming.* HL only. Command term: **describe**. Builds on A3.3.4.

*Housekeeping needs to know which rooms are occupied. Handing them a table with passport numbers is awkward and a genuine security problem. A view is the answer.*

**How to use this notebook**
- Click a code cell and press **Shift+Enter** to run it. Run the cells **in order, from the top**.
- A cell that starts with `%%sql` holds plain SQL. A cell with `<<` after `%%sql` saves its result so the check cell can look at it. A cell that creates or changes things shows an empty box. That is normal.
- A **TODO** cell is yours. Each starts with a placeholder line `SELECT 'write your SQL here' AS todo;`. Delete that line when you have written your own SQL.
- The **check** cells print ✓ when you are right and ✗ with the difference when you are not.
- The database lives in memory. If you restart the kernel, run the cells again from the top.

Notes adapted from the ExamEdge page https://examedge.app/learn/ib/a3-3-5

In [ ]:
# Helper functions for the check cells. Run this once. You do not need to read it.
def _rows(var):
    if var not in globals():
        return None
    return [tuple(r) for r in globals()[var]]

def _norm(rows, places=2):
    return [tuple(round(v, places) if isinstance(v, float) else v for v in r) for r in rows]

def _compare(name, got, want, ordered):
    g, w = _norm(got), _norm(want)
    ok = (g == w) if ordered else (sorted(g, key=repr) == sorted(w, key=repr))
    if ok:
        print("✓", name, "is correct.")
    else:
        print("✗", name, "is not right yet.")
        print("   You returned", len(g), "row(s):", g[:6])
        print("   Expected    ", len(w), "row(s):", w[:6])

def check_rows(name, var, want, ordered=False):
    # Compares the rows saved by a "%%sql var <<" cell with the expected rows.
    got = _rows(var)
    if got is None:
        print("✗", name, ": there is no result yet. Run your SQL cell first.")
        return
    _compare(name, got, want, ordered)

def check_db(name, sql, want, ordered=False):
    # Runs a question against the practice database and compares the rows.
    try:
        got = [tuple(r) for r in db.execute(sql).fetchall()]
    except Exception as e:
        print("✗", name, ": my check could not run:", e)
        return
    _compare(name, got, want, ordered)

def check_choice(choice, correct, why_right, why_wrong):
    c = str(choice).strip().upper()
    if c not in "ABCD" or c == "":
        print("Set choice to A, B, C or D, then run this cell again.")
    elif c == correct:
        print("✓ Correct.", why_right)
    else:
        print("✗ Not quite.", why_wrong)

def check_dict(name, got, want):
    bad = [k for k in want if str(got.get(k, "")).strip().upper() != want[k]]
    if not bad:
        print("✓", name, "all correct.")
    else:
        print("✗", name, ":", len(bad), "of", len(want), "not right yet:")
        for k in bad:
            print("    ", k)

In [ ]:
# Install the SQL add-on into THIS kernel if it is missing (one time only). (Given.)
import importlib.util, subprocess, sys
if importlib.util.find_spec("sql") is None:
    print("Installing the SQL add-on (one time, about a minute)...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jupysql"])
    importlib.invalidate_caches()

import sqlite3
db = sqlite3.connect(":memory:")

def reset():
    db.executescript('''
    DROP VIEW IF EXISTS RoomOccupancy; DROP VIEW IF EXISTS FrontDesk;
    DROP VIEW IF EXISTS GuestContact;  DROP VIEW IF EXISTS GuestNights;
    DROP TABLE IF EXISTS RevenueByType; DROP TABLE IF EXISTS Stay; DROP TABLE IF EXISTS Guest;
    CREATE TABLE Guest (GuestID TEXT PRIMARY KEY, GuestName TEXT, PassportNo TEXT, CardLast4 TEXT, Email TEXT);
    CREATE TABLE Stay (StayID TEXT PRIMARY KEY, GuestID TEXT, RoomNo INTEGER, RoomType TEXT, Nights INTEGER, TotalPaid REAL,
                       FOREIGN KEY (GuestID) REFERENCES Guest (GuestID));
    INSERT INTO Guest VALUES ('G1', 'Marta Oyelaran', 'X8842193', '4417', 'm.oyelaran@post.example'),
                             ('G2', 'Kenji Aluko',    'P2237741', '9052', 'k.aluko@post.example'),
                             ('G3', 'Sabine Reuter',  'T5590128', '3318', 's.reuter@post.example');
    INSERT INTO Stay VALUES ('T1', 'G1', 204, 'Double', 3, 420.00), ('T2', 'G2', 118, 'Single', 1, 95.00),
                            ('T3', 'G3', 305, 'Suite', 2, 640.00),  ('T4', 'G1', 210, 'Double', 4, 560.00);
    ''')

reset()

%load_ext sql
%config SqlMagic.displaycon = False
%config SqlMagic.feedback = False
%sql db

## 1. What a view is

A **view** looks and behaves like an ordinary table: you write `CREATE VIEW` once, then query it like any other table. Underneath, it is a **saved `SELECT` statement** that presents data from one or more real tables in whatever shape a reader needs.

> **A view stores the query, not the data.** Nothing is duplicated, so there is no second copy to keep in step. When someone queries a view, the stored `SELECT` runs against the live base tables at that moment.

Our hotel has two base tables.

In [ ]:
%%sql
SELECT * FROM Guest;

In [ ]:
%%sql
SELECT * FROM Stay;

Three staff roles need three different slices of this.

## 2. Virtual views: simple and complex

**A simple view** reads one table and only picks fields: no join and no aggregate. Housekeeping gets one.

In [ ]:
%%sql
CREATE VIEW RoomOccupancy AS
SELECT RoomNo, RoomType, Nights
FROM Stay;

In [ ]:
%%sql
SELECT * FROM RoomOccupancy;

This is the **security** benefit. Housekeeping cannot reach a passport number, because the view never selected one. No permission had to be taken away column by column.

**A complex view** uses more than one table, or something heavier than picking fields: a `JOIN`, a `GROUP BY` or an aggregate. The front desk gets one that hides the join.

In [ ]:
%%sql
CREATE VIEW FrontDesk AS
SELECT Guest.GuestName, Stay.RoomNo, Stay.Nights
FROM Stay
JOIN Guest ON Stay.GuestID = Guest.GuestID;

In [ ]:
%%sql
SELECT * FROM FrontDesk;

Two benefits at once. The receptionist types `SELECT * FROM FrontDesk;` and never writes a join (**complexity hiding** and **query simplification**), and the passport and card columns remain unreachable (**security**).

## 3. A virtual view is always current

Add a new booking and read the view again. We did nothing to the view.

In [ ]:
%%sql
INSERT INTO Stay (StayID, GuestID, RoomNo, RoomType, Nights, TotalPaid) VALUES ('T5', 'G2', 307, 'Suite', 1, 300.00);
SELECT * FROM RoomOccupancy;

Room 307 is already there, because the view re-ran its query and the row existed.

## 4. Materialized (snapshot) views

A **materialized view** breaks the rule on purpose. It runs the query **once**, **stores the result**, and serves that stored result until it is refreshed. It is the only kind of view that holds data of its own. The accountant wants revenue per room type: an expensive query nobody needs to the second.

SQLite has no `CREATE MATERIALIZED VIEW` (PostgreSQL and Oracle do), so we **build one by hand**: a real table made from the query. That is exactly what a snapshot is.

In [ ]:
%%sql
CREATE TABLE RevenueByType AS
SELECT RoomType, SUM(TotalPaid) AS Revenue
FROM Stay
GROUP BY RoomType;

In [ ]:
%%sql
SELECT * FROM RevenueByType;

That snapshot was taken **before** booking T5. Compare it with what the stored query would give if it ran now.

In [ ]:
%%sql
SELECT RoomType, SUM(TotalPaid) AS Revenue FROM Stay GROUP BY RoomType;

The Suite total in the snapshot is still **640**, but the live data says **940**. The snapshot is not broken. It shows the data as it stood at the last refresh. **Refreshing** means rebuilding it.

In [ ]:
%%sql
DROP TABLE RevenueByType;
CREATE TABLE RevenueByType AS SELECT RoomType, SUM(TotalPaid) AS Revenue FROM Stay GROUP BY RoomType;
SELECT * FROM RevenueByType;

| | Virtual view | Materialized view |
|---|---|---|
| Stores | only the query | the query result too |
| Current? | always | out of date until refreshed |
| Speed | re-runs the query every read | fast, the answer already exists |
| Storage | none extra | a real copy on disk |
| Maintenance | none | someone must refresh it |
| Choose when | the answer must be current (front desk) | the answer must be fast and slightly old is fine (monthly report) |

## 5. Read-only or updatable

A virtual view stores no data, so writing to one is really a request to write to the tables underneath. That is allowed only when the database can tell which base row each change belongs to. A view is **updatable** if:

1. It traces back to **one single table** (or another updatable view) and nothing else.
2. Every base-table field left out of the view **allows NULL**, so an insert through the view can leave those fields empty.
3. Its `SELECT` stays **simple**: no sub-query, no `GROUP BY`/`HAVING`, no `DISTINCT`, no aggregate, no join.

Every excluded feature destroys the one-to-one link between a row in the view and a row in a base table. SQLite goes further and makes **every** view read-only (unless you add special triggers). Try writing through a view:

In [ ]:
try:
    db.execute("INSERT INTO RoomOccupancy (RoomNo, RoomType, Nights) VALUES (999, 'Single', 1)")
except Exception as e:
    print("Error:", e)

## 6. Your turn

**TODO 1 (simple view).** The marketing team needs only guest names and email addresses. Create a view called `GuestContact` with `GuestName` and `Email`.

In [ ]:
%%sql
-- TODO 1: CREATE VIEW GuestContact AS SELECT ... FROM Guest;
-- Delete the next line when you have written your SQL.
SELECT 'write your SQL here' AS todo;

In [ ]:
check_db("TODO 1 columns", "SELECT name FROM pragma_table_info('GuestContact') ORDER BY cid", [("GuestName",), ("Email",)], ordered=True)
check_db("TODO 1 rows", "SELECT COUNT(*) FROM GuestContact", [(3,)])

**TODO 2 (complex view).** Create a view `GuestNights` that shows each guest's name and their total nights stayed (`GuestName`, `TotalNights`). You need a `JOIN` and a `GROUP BY`.

In [ ]:
%%sql
-- TODO 2: CREATE VIEW GuestNights AS SELECT ... JOIN ... GROUP BY ...;
-- Delete the next line when you have written your SQL.
SELECT 'write your SQL here' AS todo;

In [ ]:
check_db("TODO 2", "SELECT GuestName, TotalNights FROM GuestNights", [("Marta Oyelaran", 7), ("Kenji Aluko", 2), ("Sabine Reuter", 2)])

**TODO 3 (prove the view is live).** `GuestNights` includes T5, so Kenji has 2 nights. Add a brand-new stay, `T6`, guest `G3`, room 310, `Double`, 5 nights, 700.00, and then read `GuestNights` again to see it change without touching the view.

In [ ]:
%%sql
-- TODO 3: INSERT INTO Stay (...) VALUES (...); then SELECT * FROM GuestNights;
-- Delete the next line when you have written your SQL.
SELECT 'write your SQL here' AS todo;

In [ ]:
check_db("TODO 3", "SELECT TotalNights FROM GuestNights WHERE GuestName = 'Sabine Reuter'", [(7,)])

**TODO 4 (materialized view).** Build a snapshot table `RoomTypeTotals` with `RoomType` and `Bookings` (the count of stays for each room type). Then add one more stay and compare the snapshot with the live count.

In [ ]:
%%sql
-- TODO 4: CREATE TABLE RoomTypeTotals AS SELECT RoomType, COUNT(*) AS Bookings FROM Stay GROUP BY RoomType;
-- Delete the next line when you have written your SQL.
SELECT 'write your SQL here' AS todo;

In [ ]:
check_db("TODO 4", "SELECT RoomType, Bookings FROM RoomTypeTotals", [("Double", 3), ("Single", 1), ("Suite", 2)])
if db.execute("SELECT name FROM sqlite_master WHERE name = 'RoomTypeTotals'").fetchone():
    db.execute("INSERT INTO Stay VALUES ('T7', 'G1', 401, 'Suite', 2, 800.00)")
    print("After a new Suite booking:")
    print("   snapshot says Suite =", db.execute("SELECT Bookings FROM RoomTypeTotals WHERE RoomType = 'Suite'").fetchone()[0])
    print("   live data says  Suite =", db.execute("SELECT COUNT(*) FROM Stay WHERE RoomType = 'Suite'").fetchone()[0])

**TODO 5: updatable or read-only?** Using the three conditions, write `"UPDATABLE"` or `"READ-ONLY"` for each view (assume all omitted columns allow NULL).

In [ ]:
verdicts = {
    "CREATE VIEW V1 AS SELECT RoomNo, Nights FROM Stay;": "?",
    "CREATE VIEW V2 AS SELECT GuestName, RoomNo FROM Stay JOIN Guest ON Stay.GuestID = Guest.GuestID;": "?",
    "CREATE VIEW V3 AS SELECT RoomType, SUM(TotalPaid) FROM Stay GROUP BY RoomType;": "?",
    "CREATE VIEW V4 AS SELECT DISTINCT RoomType FROM Stay;": "?",
}
check_dict("TODO 5", verdicts, {k: v for k, v in zip(verdicts, ["UPDATABLE", "READ-ONLY", "READ-ONLY", "READ-ONLY"])})

## 7. Check your understanding

A row is updated in the base table. Which kind of view might **still show the old value** afterwards? **A** a simple view, **B** a complex view, **C** a materialized (snapshot) view, **D** any view, since all views hold their own copy of the data.

In [ ]:
choice = "?"   # type A, B, C or D
check_choice(choice, "C", "A materialized view stores its result set, so it serves the answer from the last refresh. Every other type re-runs its query and is always current.",
             "Simple and complex views are both virtual: they store only the query and re-run it on every read. Which kind keeps a stored copy that can fall behind?")

**Examiner note.** Views are often described as *"a copy of part of a table."* That is wrong for every virtual view. If a view were a copy it would need syncing, and the consistency and independence benefits would not exist. Say instead: *"A view is a stored SELECT query presented as a virtual table. It holds no data of its own, so it reads the live base tables each time. A materialized view is the exception: it stores its pre-computed result and is refreshed periodically."* Stored-query idea first, exception second.

## 8. Revision questions

**Q1. Describe what is meant by a database view.** [2 marks]

Double-click this box and type your answer here, then press Shift+Enter.

> (your answer)

**Q2. Distinguish between a virtual view and a materialized view.** [4 marks]

Double-click this box and type your answer here, then press Shift+Enter.

> (your answer)

**Q3. A hotel gives housekeeping staff access to a view rather than to the GUEST table itself. Describe two benefits of this decision, and explain why the view is likely to be read-only.** [5 marks]

Double-click this box and type your answer here, then press Shift+Enter.

> (your answer)